In [ ]:
#imports
import pandas as pd
import numpy as np
import os



In [ ]:
#open df and check if it's null
df = pd.read_csv('../data/train_val.csv')
#check if there are any null values
print(df.isnull().sum())
#print YES if it's more than zero, and the respective columns
for i in df.columns:
    if df[i].isnull().sum() > 0:
        print(i)
        #print sum
        print(df[i].isnull().sum())
#now the norm
df = pd.read_csv('../data/train_val_norm.csv')
print(df.isnull().sum())
for i in df.columns:
    if df[i].isnull().sum() > 0:
        print(i)
        print(df[i].isnull().sum())

#remove the null values
df = df.dropna()
df.to_csv('../data/train_val_norm.csv', index=False)


In [ ]:
#open df and check if it's null
df = pd.read_csv('../data/test.csv')
#check if there are any null values
print(df.isnull().sum())
#print YES if it's more than zero, and the respective columns
for i in df.columns:
    if df[i].isnull().sum() > 0:
        print(i)
        #print sum
        print(df[i].isnull().sum())
        print('NULLLLLL!')
#now the norm
#df = pd.read_csv('../data/test.csv')
print(df.isnull().sum())
for i in df.columns:
    if df[i].isnull().sum() > 0:
        print('NULL!!!')
        print(i)
        print(df[i].isnull().sum())

#remove the null values
df = df.dropna()
#df.to_csv('../data/train_val_norm.csv', index=False)


In [ ]:
import regex as re
def parse_log_file(file_path, folder_name, label_type):
    # Regex patterns to identify different sections
    type_pattern = r"------------ TYPE ------------\n(.*?)\n------------ METRICS ------------"
    metrics_pattern = r"------------ METRICS ------------\n(.*?)\n------------ PARAMETERS ------------"
    params_pattern = r"------------ PARAMETERS ------------\n(.*?)\nModel Keys:"

    # Read the file
    with open(file_path, 'r') as file:
        content = file.read()

    # Extract sections
    type_sections = re.findall(type_pattern, content, re.DOTALL)
    metrics_sections = re.findall(metrics_pattern, content, re.DOTALL)
    params_sections = re.findall(params_pattern, content, re.DOTALL)

    data = []

    # Iterate over extracted sections
    for type_sec, metrics_sec, params_sec in zip(type_sections, metrics_sections, params_sections):
        # Extract TYPE variables
        type_vars = {}
        for line in type_sec.strip().split('\n'):
            key, value = map(str.strip, line.split('='))
            type_vars[key] = value
        
        # Extract METRICS variables
        metrics_vars = {}
        for line in metrics_sec.strip().split('\n'):
            key, value = line.split(': ')
            value = value.strip('[]').split(', ')
            if ',' in str(value):
                metrics_vars[key] = list(map(float, value))
            else:
                metrics_vars[key] = float(value[0])
        
        # Extract PARAMETERS
        params_vars = {}
        for line in params_sec.strip().split('\n'):
            key, value = map(str.strip, line.split('s: '))
            params_vars[key] = value
        
        # Combine all extracted variables into a single dictionary
        combined_vars = {**type_vars, **metrics_vars, **params_vars}
        data.append(combined_vars)

    # Create a DataFrame from the extracted data
    df = pd.DataFrame(data)
    df['folder_name'] = folder_name
    df['label'] = label_type
    #move these two to be first and second columns
    cols = df.columns.tolist()
    cols = cols[-2:] + cols[:-2]
    df = df[cols]

    test_acc = df['Test Accuracy'].mean()
    test_acc_std = df['Test Accuracy'].std()
    test_prec = df['Test Precision'].mean()
    test_prec_std = df['Test Precision'].std()
    test_recall = df['Test Recall'].mean()
    test_recall_std = df['Test Recall'].std()
    test_f1 = df['Test F1'].mean()
    test_f1_std = df['Test F1'].std()
    test_tol_acc = df['Test Accuracy Tolerant'].mean()
    test_tol_acc_std = df['Test Accuracy Tolerant'].std()
    test_tol_prec = df['Test Precision Tolerant'].mean()
    test_tol_prec_std = df['Test Precision Tolerant'].std()
    test_tol_recall = df['Test Recall Tolerant'].mean()
    test_tol_recall_std = df['Test Recall Tolerant'].std()
    test_tol_f1 = df['Test F1 Tolerant'].mean()
    test_tol_f1_std = df['Test F1 Tolerant'].std()

    #put into dictionary
    avg_metrics = {'Test Accuracy': test_acc, 'Test Accuracy Std': test_acc_std, 'Test Precision': test_prec,
                    'Test Precision Std': test_prec_std, 'Test Recall': test_recall, 'Test Recall Std': test_recall_std, 
                    'Test F1': test_f1, 'Test F1 Std': test_f1_std, 'Test Accuracy Tolerant': test_tol_acc, 
                    'Test Accuracy Tolerant Std': test_tol_acc_std, 'Test Precision Tolerant': test_tol_prec, 
                    'Test Precision Tolerant Std': test_tol_prec_std, 'Test Recall Tolerant': test_tol_recall, 
                    'Test Recall Tolerant Std': test_tol_recall_std, 'Test F1 Tolerant': test_tol_f1, 
                    'Test F1 Tolerant Std': test_tol_f1_std}



    
    return df, avg_metrics

In [ ]:
#go folder by folder, open the results folder, and then check each folder too

folders_model = os.listdir()
#keep only the folders
folders_model = [folder for folder in folders_model if os.path.isdir(folder)]
#remove folders with _
folders_model = [folder for folder in folders_model if not '_' in folder]
print(folders_model)

all_models_df = pd.DataFrame()

for folder in folders_model:
    path_results = folder + '/results'
    if os.path.exists(path_results):
        print('folder:', folder)
        folders_runs = os.listdir(path_results)
        for folds in folders_runs:
            if folder == 'lstm':
                path_runs = path_results + '/' + folds + '/results_LSTM.txt'
            else:
                path_runs = path_results + '/' + folds + '/results.txt'
            if os.path.exists(path_runs):
                print('folds:', folds)
                #label type is xxx_LABEL_xxx in the name of the folder
                label_type = folds.split('_')[1]
                print(label_type)
                df, avg_m = parse_log_file(path_runs, folds, label_type)
                all_models_df = pd.concat([all_models_df, df], axis=0)

all_models_df.to_csv('all_models.csv', index=False)
all_models_df
                
            

In [ ]:
#go folder by folder, open the results folder, and then check each folder too
folder_best = './results_allfolds'
folders_model = os.listdir(folder_best)
#keep only the folders
#folders_model = [folder for folder in folders_model if os.path.isdir(folder)]
print(folders_model)

all_models_df = pd.DataFrame()

for folder in folders_model:
        path_results = folder_best + '/' + folder
        if 'LSTM' in folder:
            path_runs = path_results + '/results_LSTM.txt'
        else:
            path_runs = path_results + '/results.txt'
        folds = folder
        if os.path.exists(path_runs):
                print('folds:', folds)
                #label type is xxx_LABEL_xxx in the name of the folder
                label_type = folds.split('_')[1]
                print(label_type)
                df, avg_m = parse_log_file(path_runs, folds, label_type)
                #make avgm a dataframe with the index as the folds
                avg_m = pd.DataFrame(avg_m, index=[folds])
                all_models_df = pd.concat([all_models_df, avg_m], axis=0)

                print(all_models_df)
all_models_df.to_csv('fold_metrics.csv')
#now create df where metric and metric std are fused into single column like '$metric \pm metric std$'
all_models_df = pd.read_csv('fold_metrics.csv')
all_models_df_latex = all_models_df.copy()
for i in all_models_df.columns:
    if 'Std' in i:
        metric = i.split(' ')[:-1]
        metric = ' '.join(metric)
        #make string and round to 3 decimals
        all_models_df[i] = all_models_df[i].apply(lambda x: round(x, 3))
        all_models_df[metric] = all_models_df[metric].apply(lambda x: round(x, 3))
        for ind, row in all_models_df.iterrows():
            all_models_df_latex.loc[ind, metric] = '$' + str(row[metric]) + ' \pm ' + str(row[i]) + '$'
        all_models_df_latex = all_models_df_latex.drop(i, axis=1)
all_models_df_latex.to_csv('fold_metrics_latex.csv')
all_models_df_latex
                
            

In [ ]:
#go folder by folder, open the results folder, and then check each folder too

folders_model_final = os.listdir('../data/final_models')

path_folder = '../data/final_models/'

all_models_df = pd.DataFrame()

for folder in folders_model_final:
    path_results = path_folder + folder + '/results.txt'
    if os.path.exists(path_results):
        #label type is xxx_LABEL_xxx in the name of the folder
        label_type = folder.split('_')[1]
        print(label_type)
        df = parse_log_file(path_results, folder, label_type)
        all_models_df = pd.concat([all_models_df, df], axis=0)

all_models_df.to_csv('all_final_models.csv', index=False)
all_models_df
                
            

In [ ]:
#impors keras libraries and get_metrics to load a odel and its weights and make predictios
from keras.models import load_model
from get_metrics import get_metrics
from createDataSplits import createDataSplits_test
import pandas as pd

#select winning folders
folds_final = ['gru_rm_norm_all3_2024-05-28_00-55-36','bilstm_UA_norm_all1_2024-05-28_01-00-45','bilstm_IR_all2_2024-05-28_01-17-00']
labels = ['ua','rm','ir']

df_test_norm = pd.read_csv('../data/test_norm.csv')
df_test = pd.read_csv('../data/test.csv')

sequence_length = 5



#get the models
for label in labels:
    model = load_model('../data/final_models/models'  + '/model' + label+'.h5')
    model.load_weights('../data/final_models/models'  + '/model' + label + 'weights.h5')
    
    print('LABEL:', label)

    if label == 'ua':
        lab = 'UserAwkwardness'
        label_column = 'UserAwkwardness'
    elif label == 'rm':
        lab = 'RobotMistake'
        label_column = 'RobotMistake'
    elif label == 'ir':
        lab = 'InteractionRupture'
        label_column = 'InteractionRupture'
    num_classes = np.unique(df_test[lab]).shape[0]

    if label == 'ir':
        sessions, X_test, y_test, X_test_sequences, y_test_sequences = createDataSplits_test(df_test, num_classes, label_column = label_column, results_directory= '../logs/', seed_value = 42, sequence_length = sequence_length)
    else:
        sessions, X_test, y_test, X_test_sequences, y_test_sequences = createDataSplits_test(df_test_norm, num_classes, label_column = label_column, results_directory= '../logs/', seed_value = 42, sequence_length = sequence_length)

    y_predict_probs = model.predict(np.array(X_test_sequences))
    y_pred = np.argmax(y_predict_probs, axis=1)  # Convert to class labels
    #if fold_no == 1:
    #    y_pred_prev = y_pred
    #see if y_pred == y_pred_prev
    print('y_pred:', y_pred)
    #print('y_pred_prev:', y_pred_prev)
    #print('y_pred == y_pred_prev:', y_pred == y_pred_prev)
    #y_pred_prev = y_pred
    #print('y_pred shape:', y_pred.shape)
    #transform y_test sequences so single array, from one-hot
    y_test_sequences = np.argmax(y_test_sequences, axis=1)
    print('sessions shape:', len(sessions))

    #calculate metrics
    test_metrics = get_metrics(y_pred, np.array(y_test_sequences), sessions = sessions, tolerance=1)
    print('test metrics:')
    print(test_metrics)

